In [12]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))

assert_files_exist(
  "HLA"  = paste0(LILAC_DIR, "lilac.csv"),
  "Linx drivers"     = paste0(LINX_DIR, "driver.catalog.csv"),
  "PURPLE drivers"     = paste0(PURPLE_DIR, "driver.catalog.somatic.csv")
)

# 0 - Prepare lilac

In [33]:
lilac <- fread( paste0(LILAC_DIR, "lilac.csv"))

In [78]:
hla_loh <-
inner_join(
 lilac %>% group_by(sampleId) %>% summarise(x = n_distinct(Allele)),
 lilac %>% filter(TumorCopyNumber > .5) %>% group_by(sampleId) %>% summarise(y = n_distinct(Allele)),
 by = "sampleId") %>%
 mutate( hla_loh = as.numeric(x - y > 0) ) %>%
 select( sampleId, hla_loh )

# 1 - Prepare Drivers

In [79]:
drivers <- fread(paste0(LINX_DIR, "driver.catalog.csv")) 

In [80]:
drivers_ready <- 
drivers %>% 
  filter(driverLikelihood > .8) %>% 
  transmute(sampleId, gene = paste0("driver_", gene)) %>% 
  unique() %>% 
  mutate(driver = 1) %>% 
  spread(gene, driver)


In [81]:
drivers_go <-
drivers_ready %>%
 select(sampleId, driver_POLE, driver_POLD1, driver_KEAP1, driver_STK11, driver_EGFR, driver_JAK1, driver_JAK2, driver_B2M) 

# 2 - Together

In [82]:
extra_markers <- 
hla_loh %>%
 left_join(drivers_ready, by = "sampleId")
extra_markers[is.na(extra_markers)] <- 0

In [83]:
saveRDS(extra_markers, paste0(CIDER_PROJ_DIR, "extra_markers.rds"))